# 3D Analysis of an Extended Source

In this example we fit the spectum and spatial extension of RX J1713.7-3946 (called rxj1713 in the following because typing is hard) first jointly and then stacked

## Joint Fit

Load all the relevant modules and set the energy unit to `u.TeV` as we are only dealing with H.E.S.S. data here

Additionally we load the `Gaussian_on_sphere` from `gammapy_plugin.utils.astromodels_functions` and `Cutoff_powerlaw` from `astromodels` for modelling the source later

In [ ]:
%matplotlib inline
import astropy.units as u
import numpy as np
import matplotlib.pyplot as plt
from astromodels.core.model import Model
from astromodels.core.units import get_units
from astromodels.functions import (
    Log_uniform_prior,
    Cutoff_powerlaw,
    Uniform_prior,
)
from astromodels.sources.extended_source import ExtendedSource
from astropy.coordinates import SkyCoord
from gammapy.data import DataStore
from gammapy.datasets import MapDataset, Datasets
from gammapy.makers import (
    FoVBackgroundMaker,
    MapDatasetMaker,
    SafeMaskMaker,
)
from gammapy.maps import MapAxis, WcsGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    PowerLawSpectralModel,
    SkyModel,
    GaussianSpatialModel,
    FoVBackgroundModel,
)
from regions import CircleSkyRegion
from threeML import DataList, load_analysis_results
from threeML.bayesian.bayesian_analysis import BayesianAnalysis
from threeML.classicMLE.joint_likelihood import JointLikelihood
from threeML.utils.progress_bar import trange

from gammapy_plugin.converter import AstromodelConverter
from gammapy_plugin.GammapyLike import GammapyLike
from gammapy_plugin.utils.astromodels_functions import Gaussian_on_sphere

get_units().energy = u.TeV

Select all the observations within a `5 deg` radius around RX J1713.7-3946

Prepare the geometry by setting the energy axis and `WcsGeom`

In [ ]:
datastore = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1/")
target_position = SkyCoord.from_name("RX J1713.7-3946").galactic

selection = dict(
    type="sky_circle",
    frame="galactic",
    lon=target_position.l,
    lat=target_position.b,
    radius="5deg",
)
select_obs_tab = datastore.obs_table.select_observations(selection)

obs = datastore.get_observations(select_obs_tab["OBS_ID"])

# Prepare the geometry
energy_axis = MapAxis.from_energy_bounds(0.2, 40.0, 10, per_decade=True, unit="TeV")
energy_axis_true = MapAxis.from_energy_bounds(
    0.1, 100, 20, per_decade=True, unit="TeV", name="energy_true"
)
geom = WcsGeom.create(
    skydir=target_position,
    binsz=0.02,
    width=(6 * u.deg, 6 * u.deg),
    frame="galactic",
    axes=[energy_axis],
)

Create the relevant `Makers`

We will also run the `FoVBackgroundMaker` when jointly fitting - this is not needed

We exclude a `1 deg` circle at the source poition for fitting the background models to reduce overestimation of the background


In [ ]:
circle = CircleSkyRegion(center=target_position, radius=1 * u.deg)
regions = [circle]
exclusion_mask = ~geom.region_mask(regions=regions)
maker = MapDatasetMaker(
    selection=["counts", "background", "psf", "edisp", "exposure"],
)
safe_mask_maker = SafeMaskMaker(
    methods=["offset-max", "aeff-max", "bkg-peak"], offset_max="2.3 deg"
)
fov_bkg_maker = FoVBackgroundMaker(method="fit", exclusion_mask=exclusion_mask)

And now the datasets as well as the `GammapyPlugin` instances:

Few noteworthy things:
- we set the `FoVBackgroundModel` before and provide a custom `name`. As mentioned before, the background model is fitted afterwards and running the `FoVBackgroundMaker` is not necessary, but here we also print the fitted values from the `FoVBackgroundMaker` to compare to our results later
- in case you want to fit the background before and fix it during the fit you will always have to set the model before providing a non-default name as it defaults to `dataset_name + "-bkg"` which is not compatible with `astromodels`
- the `exclusion_mask` is only used for the `FoVBackgroundMaker` and will _not_ be used later during the acutal sampling
- the used map size of `6deg x 6deg` is fairly large and there are other sources in this region which we do not mask for simplicity: **_This will lead to overestimation of the background and therefore an underestimation of the source's flux!_**
- we set the dataset using default `mode="individual"` as we use on plugin per instance, afterwards we add the corresponding background model, which then gets parsed by `gammapy_plugin` and the normalization is treated as a nuisance parameter
- we used a galactic `WcsGeom` frame and we therefore also use galactic in the plugin

In [ ]:
datasets = Datasets()
gls = []
for o in obs:
    dataset = MapDataset.create(
        geom=geom, energy_axis_true=energy_axis_true, name=f"HESS_{o.obs_id}"
    )
    dataset = maker.run(dataset, o)
    dataset = safe_mask_maker.run(dataset, o)
    bkg_model = FoVBackgroundModel(name=f"{o.obs_id}_bkg", dataset_name=dataset.name)
    dataset.models = [bkg_model]
    dataset = fov_bkg_maker.run(dataset)
    print(
        f"Bkg norm for HESS_{o.obs_id}: {round(bkg_model.parameters['norm'].value,3)} +/- {round(bkg_model.parameters['norm'].error,3)}"
    )
    datasets.append(dataset)
    gl = GammapyLike(dataset.name, frame="galactic")
    gl.set_datasets(dataset)
    gl.set_background_models(bkg_model)
    gls.append(gl)

## Setting up the Model

Using a exponential cutoff-powerlaw spectral model and a 2D Gaussian spatial one for simplicity

In [ ]:
cpl = Cutoff_powerlaw()
spat = Gaussian_on_sphere(
    lon0=target_position.transform_to("galactic").l.deg,
    lat0=target_position.transform_to("galactic").b.deg,
    sigma=0.5,
)
es = ExtendedSource(source_name="rxj1713", spectral_shape=cpl, spatial_shape=spat)
cpl.K = 5e-15 * u.Unit("TeV-1 cm-2 s-1")
cpl.index = -2 * u.dimensionless_unscaled
cpl.xc = 15 * u.TeV
cpl.piv = 1 * u.TeV
cpl.piv.free = False
cpl.K.prior = Log_uniform_prior(lower_bound=1e-15, upper_bound=1e-14)
cpl.index.prior = Uniform_prior(lower_bound=-3, upper_bound=-1)
cpl.xc.prior = Uniform_prior(lower_bound=1, upper_bound=30)
spat.lon0.free = False
spat.lat0.free = False
spat.sigma.free = True
spat.sigma.prior = Uniform_prior(lower_bound=0.1, upper_bound=1)

model = Model(es)

We can either convert the model before setting it or a `AstromodelConverter` will be created when adding the model to plugins

In [ ]:
conv = AstromodelConverter(model, frame="galactic")

Now adding the model and the converted one to the plugins and specificly including the source. All sources are used by default, here we set it explicitly for no specific reason

In [ ]:
for gl in gls:
    gl.set_sources("rxj1713")
    gl.set_model(model, conv)

This next cell might take hours or days to finish using only a single core. Just skip it and load the result provided in the `result.fits` in the next cell.

```python
ba =BayesianAnalysis(model,DataList(*gls))
ba.set_sampler("multinest")
ba.sampler.setup()

res = ba.sample(quiet=False)
```

In [ ]:
res = load_analysis_results("data/result.fits")
res.display()

Take a look at the corner plot of the source's free paramters:

In [ ]:
fig = res.corner_plot(
    components=[
        "rxj1713.Gaussian_on_sphere.sigma",
        "rxj1713.spectrum.main.Cutoff_powerlaw.K",
        "rxj1713.spectrum.main.Cutoff_powerlaw.index",
        "rxj1713.spectrum.main.Cutoff_powerlaw.xc",
    ]
)
fig.show()

Lets take a look at the differential flux at 1 TeV by integrating over the spatial part - the units of the normalzation `K` are actually `cm-2 TeV-1 s-1 deg-2` and not as stated in the results `cm-2 TeV-1 s-1`

In [ ]:
flux_1tev = res.optimized_model.extended_sources["rxj1713"].spectrum.main.shape(
    1 * u.TeV
) * res.optimized_model.extended_sources[
    "rxj1713"
].spatial_shape.get_total_spatial_integral(
    z=1 * u.TeV
)


samples = res.samples
n_samples = 1000
vals = np.zeros(n_samples)
cpl_copy = Cutoff_powerlaw()
spat_copy = Gaussian_on_sphere(
    lon0=target_position.transform_to("galactic").l.deg,
    lat0=target_position.transform_to("galactic").b.deg,
    sigma=0.5,
)
es_copy = ExtendedSource(
    source_name="dummy", spectral_shape=cpl_copy, spatial_shape=spat_copy
)

for i in trange(n_samples):
    n = np.random.choice(np.arange(samples.shape[-1]))
    spat_copy.sigma.value = samples[0, n]
    cpl_copy.K.value = samples[1, n]
    cpl_copy.index.value = samples[2, n]
    cpl_copy.xc.value = samples[3, n]
    vals[i] = cpl_copy(1) * spat_copy.get_total_spatial_integral()
hpd_1tev = np.array((np.percentile(vals, 2.5), np.percentile(vals, 97.5)))
print(f"Flux: {flux_1tev} +/- {np.abs((hpd_1tev-flux_1tev.value)[::-1])}")
print(f"95% hpd interval: {hpd_1tev}")

Keeping in mind that we are overestimating the background for sure, this is on the order of the value from [H.E.S.S. Collaboration, 2018](http://dx.doi.org/10.1051/0004-6361/201629790), who determined a flux of $2.3\pm0.1\:\mathrm{e}{-11}$ cm-2 s-1 TeV-1



In [ ]:
plt.matshow(res.get_correlation_matrix())
ticks = [
    i.replace("rxj1713.", "")
    .replace("_bkg_norm", "")
    .replace("spectrum.main.Cutoff_powerlaw.", "")
    .replace("HESS_", "")
    .replace("Gaussian_on_sphere.", "")
    .split("_")[0]
    for i in res.optimized_model.free_parameters.keys()
]
plt.xticks(np.arange(19), ticks, rotation=90)
plt.yticks(np.arange(19), ticks)
plt.colorbar()

In [ ]:
datasets_temp = Datasets()
for gl in gls:
    datasets_temp.append(gl.datasets[0])

Taking a look at the energy integrated counts map of these 15 observations, a 2D-Gaussian might not be the best description and a Powerlaw neither ;)

In [ ]:
datasets.stack_reduce().counts.sum_over_axes(keepdims=False).smooth(0.02 * u.deg).plot()

## Stacking the Dataset

We can also speed things a lot by
1. stacking the datasets
2. only fitting the FoVBackgroundModels before hand and fixing their values


In [ ]:
datasets_stacked = Datasets()

for o in obs:
    dataset = MapDataset.create(
        geom=geom, energy_axis_true=energy_axis_true, name=f"HESS_{o.obs_id}"
    )
    dataset = maker.run(dataset, o)
    dataset = safe_mask_maker.run(dataset, o)
    bkg_model = FoVBackgroundModel(name=f"{o.obs_id}_bkg", dataset_name=dataset.name)
    dataset.models = [bkg_model]
    dataset = fov_bkg_maker.run(dataset)
    datasets_stacked.append(dataset)

Just stack them before passing them to the `GammapyLike` instance or set the `mode="stacked"` so the plugin does it for you :) 

In [ ]:
gl_stacked = GammapyLike("stacked", frame="galactic")
gl_stacked.set_datasets(datasets_stacked, mode="stacked")

Just the same model as before 

In [ ]:
cpl_stacked = Cutoff_powerlaw()
spat_stacked = Gaussian_on_sphere(
    lon0=target_position.transform_to("galactic").l.deg,
    lat0=target_position.transform_to("galactic").b.deg,
    sigma=0.5,
)
es_stacked = ExtendedSource(
    source_name="rxj1713_stacked",
    spectral_shape=cpl_stacked,
    spatial_shape=spat_stacked,
)
cpl_stacked.K = 5e-15 * u.Unit("TeV-1 cm-2 s-1")
cpl_stacked.index = -2 * u.dimensionless_unscaled
cpl_stacked.xc = 15 * u.TeV
cpl_stacked.piv = 1 * u.TeV
cpl_stacked.piv.free = False
cpl_stacked.K.prior = Log_uniform_prior(lower_bound=1e-15, upper_bound=1e-14)
cpl_stacked.index.prior = Uniform_prior(lower_bound=-3, upper_bound=-1)
cpl_stacked.xc.prior = Uniform_prior(lower_bound=1, upper_bound=30)
spat_stacked.lon0.free = False
spat_stacked.lat0.free = False
spat_stacked.sigma.free = True
spat_stacked.sigma.prior = Uniform_prior(lower_bound=0.1, upper_bound=1)

model_stacked = Model(es_stacked)
conv_stacked = AstromodelConverter(model_stacked, "galactic")

In [ ]:
gl_stacked.set_model(model_stacked, conv_stacked)

For simplicity just use minuit to minimize the likelihood

In [ ]:
jl = JointLikelihood(model_stacked, DataList(gl_stacked))

In [ ]:
import time

In [ ]:
start = time.time()
jl.fit()
stop = time.time()
print(f"Fit finished in {round(stop-start,3)}s")
res_stacked = jl.results

In [ ]:
flux_1tev_stacked = res_stacked.optimized_model.extended_sources[
    "rxj1713_stacked"
].spectrum.main.shape(1 * u.TeV) * res_stacked.optimized_model.extended_sources[
    "rxj1713_stacked"
].spatial_shape.get_total_spatial_integral(
    z=1 * u.TeV
)

print(f"Flux: {flux_1tev_stacked}")

Compared to the multiple hours `multinest` took to sample the full posterior using 10 threads this is way way faster at the cost of all the disadvantages of stacking the data ;)